# Production Chat Debug

Replikuje dokładnie ścieżkę produkcyjną `/chat` (routes.py + prompt_builder_v2).  
Przy każdym pytaniu drukuje pełny kontekst wysyłany do LLM: `system` + `messages`.  
Obsługuje wieloturową rozmowę (historia w pamięci komórki).

In [1]:
from __future__ import annotations

import os
import sys
import uuid
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from backend.app.config import (
    CHAT_MODEL,
    V2_CHUNK_MAX_CHARS,
    V2_CHUNK_OVERLAP_CHARS,
    V2_TOP_K,
    VISION_CACHE_DIR,
)
from backend.app.conversation.llm_client import ask
from backend.app.conversation.prompt_builder_v2 import (
    SYSTEM_PROMPT,
    build_prompt_v2,
    match_sources,
    parse_citations,
)
from backend.app.document.chunker_v2 import chunk_document
from backend.app.document.models import (
    DocumentMetadata,
    ExtractedChapter,
    ExtractedDocument,
)
from backend.app.retrieval.vector_store_v2 import index_v2_chunks, reset_collection, search_v2

print(f"ROOT: {ROOT}")
print(f"Model: {CHAT_MODEL}")
print(f"Chunk max chars: {V2_CHUNK_MAX_CHARS}, overlap: {V2_CHUNK_OVERLAP_CHARS}")
print(f"Top-K: {V2_TOP_K}")

ROOT: c:\mirror\zadanie\docquery
Model: claude-sonnet-4-20250514
Chunk max chars: 800, overlap: 0
Top-K: 5


In [2]:
# Indeksowanie — dokładna replika lifespan() z main.py
# reset_collection() zapobiega duplikatom przy ponownym uruchomieniu komórki
CHAPTER_ORDER = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]

chapters = [
    ExtractedChapter.load_json(VISION_CACHE_DIR / f"{cid}.json")
    for cid in CHAPTER_ORDER
    if (VISION_CACHE_DIR / f"{cid}.json").exists()
]

assert chapters, f"Brak rozdziałów w {VISION_CACHE_DIR} — uruchom ekstrakcję vision."

doc = ExtractedDocument(
    metadata=DocumentMetadata(
        source_file="raport_2024_pl.pdf",
        total_pages=sum(len(c.pages) for c in chapters),
        extraction_date="cache",
    ),
    title="BGK 2024",
    chapters=chapters,
)

chunks = chunk_document(doc, max_chars=V2_CHUNK_MAX_CHARS, overlap_chars=V2_CHUNK_OVERLAP_CHARS)

reset_collection()
index_v2_chunks(chunks)

print(f"Zaindeksowano {len(chunks)} chunków z {len(chapters)} rozdziałów.")

Zaindeksowano 802 chunków z 10 rozdziałów.


In [3]:
# Stan konwersacji — uruchom tę komórkę ponownie żeby zresetować historię
history: list[dict] = []
session_id = uuid.uuid4().hex
print(f"session_id: {session_id}")


def chat_debug(question: str) -> str:
    """Replika /chat z pełnym wydrukiem kontekstu wysyłanego do LLM."""
    retrieved = search_v2(question, top_k=V2_TOP_K)
    system, messages = build_prompt_v2(question, retrieved, history)

    SEP = "=" * 70
    THIN = "-" * 70

    print(SEP)
    print("SYSTEM PROMPT:")
    print(THIN)
    print(system)
    print(SEP)
    print(f"MESSAGES ({len(messages)} szt.):")
    for i, msg in enumerate(messages):
        print(f"\n[{i}] role={msg['role']}")
        print(THIN)
        print(msg["content"])
    print(SEP)

    answer = ask(messages, system)

    # Kolejność jak w routes.py: najpierw call do API, potem zapis historii
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})

    citations = parse_citations(answer)
    sources = match_sources(retrieved, citations)

    print("\nODPOWIEDŹ:")
    print(answer)

    if sources:
        print(f"\nŹRÓDŁA ({len(sources)}):")
        for s in sources:
            pages_str = (
                f"s. {s['pages'][0]}"
                if len(s["pages"]) == 1
                else f"s. {s['pages'][0]}–{s['pages'][-1]}"
            )
            chapter = s["chapter"] or ""
            section = s["section"] or ""
            location = " › ".join(x for x in [chapter, section] if x)
            print(f"  [{s['element_type']}] {pages_str} — {location}")

    return answer

session_id: e9588147965d42adbd571f6b5fabd284


In [4]:
# Pytanie 1
_ = chat_debug("Jakie były przychody BGK w 2024 roku?")

SYSTEM PROMPT:
----------------------------------------------------------------------
Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonych fragmentów dokumentu.

Zasady:
1. Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych fragmentów. Nie korzystaj z wiedzy zewnętrznej.
2. Każdą tezę potwierdź dosłownym cytatem z dokumentu w cudzysłowie.
3. Przy każdym cytacie podaj numer strony lub zakres stron:
   - Jeśli cytat pochodzi z jednej strony: [Strona X]
   - Jeśli cytat obejmuje kilka kolejnych stron: [Strony X-Y]
4. Fragmenty oznaczone jako TABELA lub INFOGRAFIKA traktuj jako dane pomocnicze.
   NIE cytuj ich dosłownie. Zamiast tego opisz zawarte w nich informacje własnymi słowami i oznacz odwołanie w formacie:
   - [Tabela, s. X] lub [Infografika, s. X]
5. Jeśli informacji nie ma w dostarczonym kontekście — powiedz o tym wprost. Nie zgaduj i nie konfabuluj.
6. Jeśli pytanie odwołuje się do wcześniejszej rozmowy (np. 'to', 'a rok wcześniej', 'ten dokument'), wyko

In [ ]:
# # Pytanie 2 — test konwersacji (model powinien widzieć historię w MESSAGES)
# _ = chat_debug("A jak to się zmieniło rok do roku?") #nie odpowie poprawnie bo rag wyszkia fragmentó dosłownie dla tego pytania

SYSTEM PROMPT:
----------------------------------------------------------------------
Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonych fragmentów dokumentu.

Zasady:
1. Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych fragmentów. Nie korzystaj z wiedzy zewnętrznej.
2. Każdą tezę potwierdź dosłownym cytatem z dokumentu w cudzysłowie.
3. Przy każdym cytacie podaj numer strony lub zakres stron:
   - Jeśli cytat pochodzi z jednej strony: [Strona X]
   - Jeśli cytat obejmuje kilka kolejnych stron: [Strony X-Y]
4. Fragmenty oznaczone jako TABELA lub INFOGRAFIKA traktuj jako dane pomocnicze.
   NIE cytuj ich dosłownie. Zamiast tego opisz zawarte w nich informacje własnymi słowami i oznacz odwołanie w formacie:
   - [Tabela, s. X] lub [Infografika, s. X]
5. Jeśli informacji nie ma w dostarczonym kontekście — powiedz o tym wprost. Nie zgaduj i nie konfabuluj.
6. Jeśli pytanie odwołuje się do wcześniejszej rozmowy (np. 'to', 'a rok wcześniej', 'ten dokument'), wyko

In [6]:
# Reset historii — uruchom żeby zacząć nową rozmowę
history.clear()
print(f"Historia wyczyszczona. Turów w historii: {len(history)}")

Historia wyczyszczona. Turów w historii: 0
